# 英語コーパス(009 スケーリング則用)の取得と Hugging Face Hub への切り出し

## 目的

[009. スケーリング則](https://github.com/kojikojiprg/ai-theories/blob/main/theories/02_pretraining/009_scaling_laws.ipynb)
の学習グリッドで使う英語コーパス(1000 記事、`en_009_scaling.json`)を取得し、
Hugging Face Hub の Dataset リポジトリ(`kojikojiprg/ai-theories-corpus-en-009-scaling`)
として保存する。

**背景**: 009 の Colab 本番実行が、コーパス取得の途中で`KeyError: 'parse'`(HTTP
ステータスは 200 だが応答 JSON に`"parse"`キーがないケースが未対応だった)を出して
停止した。この不具合は`src/data/text.py`の`_fetch_wikipedia_revision_plaintext()`・
`load_wikipedia_corpus()`側で修正済みである(記事単位の失敗を再試行対象にし、
再試行してもなお失敗する記事はスキップして全体は継続する)。

加えて、1000 記事の取得コストは毎回大きい(前回実測で約 60〜100 分)。一度取得した
コーパスを Hugging Face Hub の Dataset として保存しておけば、以降 009 を再実行する
たびに Wikipedia API から直接再取得する必要がなくなる。

**対象外**: 006 のマニフェスト・ノートブック自体は変更しない。009 側のコーパス取得
セルの更新は本ノートブックとは別に行う(依頼の 3 節)。

In [1]:
# 環境セットアップ(Google Colab)
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    !git clone https://github.com/kojikojiprg/ai-theories.git
    %cd ai-theories
    !pip install uv -q
    !uv pip install --system -r requirements.txt
# ローカル(Jupyter)実行時は、リポジトリルートで起動していればそのまま動く。

In [2]:
import json
import subprocess
import time
from pathlib import Path

from src.data.text import load_english_scaling_corpus, split_train_val_text

DRY_RUN = True  # Claude Code はこの True 側のみ実行する(Colab で DRY_RUN=False に切り替えるとアップロードが実行される)

ROOT = Path(".")
# 009 本体(theories/02_pretraining/009_scaling_laws.ipynb)と同じキャッシュディレクトリを
# 指定する。記事ごとのキャッシュ(wikipedia_en_articles/)を共有できる。
CACHE_DIR = ROOT / ".cache" / "009_corpus"
OUTPUT_DIR = ROOT / ".cache" / "fetch_and_upload_corpus_en_009"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

VALIDATION_RATIO = 0.05  # 009 と同一
CORPUS_REPO_ID = "kojikojiprg/ai-theories-corpus-en-009-scaling"
MANIFEST_NAME = "en_009_scaling.json"
MANIFEST_ARTICLE_COUNT = 1000

## コーパスの取得

`load_english_scaling_corpus()`(1 節で修正済み、`return_metadata=True`で取得結果の
メタデータも受け取る)を、記事数を縮小せず本番のマニフェスト全体(1000 記事)で呼び出す。
BPE の学習・言語モデルの学習を伴わない純粋な取得処理であり GPU を要しないため、本番
スケールのままローカルで実行する。

In [3]:
t0 = time.time()
raw_text, fetch_metadata = load_english_scaling_corpus(CACHE_DIR, return_metadata=True)
elapsed = time.time() - t0
print(f"取得時間: {elapsed:.1f} s ({elapsed / 60:.1f} 分)")

raw_bytes = len(raw_text.encode("utf-8"))
print(f"raw_text: {len(raw_text):,} 文字 / {raw_bytes:,} バイト")
print(
    f"取得できた記事数: {fetch_metadata['fetched_article_count']} / "
    f"{fetch_metadata['manifest_article_count']}"
)

if fetch_metadata["skipped_articles"]:
    print(f"[警告] {len(fetch_metadata['skipped_articles'])} 記事の取得に失敗した:")
    for a in fetch_metadata["skipped_articles"]:
        print(f"  - {a['title']}: {a['reason']}")
else:
    print("すべての記事を取得できた(スキップなし)")

assert fetch_metadata["manifest_article_count"] == MANIFEST_ARTICLE_COUNT, (
    f"マニフェスト記事数が期待({MANIFEST_ARTICLE_COUNT})と一致しない: "
    f"{fetch_metadata['manifest_article_count']}"
)

取得時間: 0.1 s (0.0 分)
raw_text: 60,247,276 文字 / 60,580,536 バイト
取得できた記事数: 999 / 1000
[警告] 1 記事の取得に失敗した:
  - List of foreign footballers in top leagues of former Yugoslavia: RuntimeError("en revid=1370486518: 応答 JSON に 'parse' キーがない(error={'code': 'nosuchrevid', 'info': 'There is no revision with ID 1370486518.', 'docref': 'See https://en.wikipedia.org/w/api.php for API usage. Subscribe to the mediawiki-api-announce mailing list at &lt;https://lists.wikimedia.org/postorius/lists/mediawiki-api-announce.lists.wikimedia.org/&gt; for notice of API deprecations and breaking changes.'})")


## 訓練・検証分割

009 と同じ`VALIDATION_RATIO=0.05`・`split_train_val_text()`で分割する。この関数は
乱数を使わず、テキスト末尾から`validation_ratio`分を切り出すだけの決定的な処理である
ことを、同じ入力に対して 2 回呼び出して結果が完全一致することで確認する。009 本体側でも
同一の`raw_text`(Hub からダウンロードした`corpus.json`の`raw_text`フィールド)に対して
同じ関数で分割するため、この結果は 009 が行う分割と完全に一致する。

In [4]:
train_text, val_text = split_train_val_text(raw_text, VALIDATION_RATIO)

train_text_2, val_text_2 = split_train_val_text(raw_text, VALIDATION_RATIO)
assert train_text == train_text_2 and val_text == val_text_2, "分割が決定的でない"
print("[OK] split_train_val_text の決定性を確認した(同じ入力から常に同じ分割になる)")

print(f"train_text: {len(train_text):,} 文字, val_text: {len(val_text):,} 文字")

[OK] split_train_val_text の決定性を確認した(同じ入力から常に同じ分割になる)
train_text: 57,234,913 文字, val_text: 3,012,363 文字


## アップロード用 JSON の組み立て

In [5]:
source_commit = subprocess.run(
    ["git", "rev-parse", "HEAD"], capture_output=True, text=True, check=True
).stdout.strip()

corpus_payload = {
    "language": "en",
    "manifest": MANIFEST_NAME,
    "manifest_article_count": fetch_metadata["manifest_article_count"],
    "fetched_article_count": fetch_metadata["fetched_article_count"],
    "skipped_articles": fetch_metadata["skipped_articles"],
    "raw_text": raw_text,
    "validation_ratio": VALIDATION_RATIO,
    "raw_bytes": raw_bytes,
    "source_commit": source_commit,
}

corpus_json_path = OUTPUT_DIR / "corpus.json"
corpus_json_path.write_text(json.dumps(corpus_payload, ensure_ascii=False), encoding="utf-8")
print(
    f"corpus.json を書き出した: {corpus_json_path} "
    f"({corpus_json_path.stat().st_size:,} バイト)"
)

corpus.json を書き出した: .cache/fetch_and_upload_corpus_en_009/corpus.json (61,398,950 バイト)


## データセットカードの作成

由来トピック(009)へのリンク・マニフェスト名・記事数・取得できなかった記事(あれば)・
ライセンスを記載する。ライセンスは`cc-by-sa-4.0`を採用する(トークナイザのモデルカードで
使った`cc-by-nc-4.0`とは異なる。理由は本文中に明記する)。

In [6]:
_skipped_section = (
    "\n".join(f"- {a['title']}: {a['reason']}" for a in corpus_payload["skipped_articles"])
    if corpus_payload["skipped_articles"]
    else "なし(すべての記事を取得できた)"
)

dataset_card = f"""---
language: en
license: cc-by-sa-4.0
tags:
- ai-theories
- corpus
- wikipedia
---

# ai-theories 英語コーパス(009 スケーリング則用)

`ai-theories`(https://github.com/kojikojiprg/ai-theories)プロジェクトの成果物。
[009. スケーリング則](https://github.com/kojikojiprg/ai-theories/blob/main/theories/02_pretraining/009_scaling_laws.ipynb)
の学習グリッドで使う英語コーパス。

## ライセンスについての注記

このデータセット自体のライセンスは **クリエイティブ・コモンズ 表示-継承 4.0 国際
(CC BY-SA 4.0)** である。`ai-theories`の他の成果物(トークナイザなど)は
`cc-by-nc-4.0`を採用しているが、本データセットの内容はフリー百科事典
『ウィキペディア(Wikipedia)』の本文そのもの(wikitext を平文に変換したのみで、
内容は改変していない)であり、Wikipedia 本文自体のライセンス(CC BY-SA 4.0、
表示・継承の条件)を継承する必要があるため、別のライセンスとしている。

## 由来

`src/data/wikipedia_manifests/{corpus_payload["manifest"]}`
(記事タイトル・リビジョン ID を固定したマニフェスト、
{corpus_payload["manifest_article_count"]} 記事)から、Wikimedia API
(`action=parse`、`oldid`でリビジョンを指定)で取得した。タイトルとリビジョン ID を
両方固定しているため、取得時点によらず同一の入力が得られる。

- マニフェスト記事数: {corpus_payload["manifest_article_count"]}
- 取得できた記事数: {corpus_payload["fetched_article_count"]}

**取得できなかった記事**:

{_skipped_section}

研究・教育目的で構築したものであり、品質保証は行っていない。商用・実運用での利用は
想定しない。

## 構成

`corpus.json` は以下のフィールドを含む。

- `raw_text`: 取得できた記事本文を連結した全文
- `validation_ratio`: {corpus_payload["validation_ratio"]}(`src/data/text.py`の
  `split_train_val_text()`で使う訓練・検証分割の比率)
- `raw_bytes`: `raw_text`の UTF-8 バイト数({corpus_payload["raw_bytes"]:,})
- `source_commit`: 取得時点の`ai-theories`リポジトリのコミットハッシュ
  (`{corpus_payload["source_commit"]}`)
"""

dataset_card_path = OUTPUT_DIR / "README.md"
dataset_card_path.write_text(dataset_card, encoding="utf-8")
print(f"データセットカードを書き出した: {dataset_card_path}")

データセットカードを書き出した: .cache/fetch_and_upload_corpus_en_009/README.md


## アップロード(Google Colab で `DRY_RUN=False` として実行する)

`DRY_RUN=True` の間はアップロードを一切呼び出さない。`DRY_RUN=False and IN_COLAB`
の場合のみ、`google.colab.userdata.get("HF_TOKEN")` で取得したトークンを使って
呼び出す(008・トークナイザ切り出しと同じ認証パターン)。`repo_type="dataset"` で
Dataset リポジトリとして作成する。Claude Code はこのセルにおいて、トークンの入力・
環境変数への設定・実際のアップロード実行を一切行わない。

In [7]:
if not DRY_RUN and IN_COLAB:
    from google.colab import userdata
    from huggingface_hub import HfApi

    token = userdata.get("HF_TOKEN")
    api = HfApi(token=token)
    api.create_repo(repo_id=CORPUS_REPO_ID, repo_type="dataset", exist_ok=True, private=False)
    api.upload_file(
        path_or_fileobj=str(corpus_json_path),
        path_in_repo="corpus.json",
        repo_id=CORPUS_REPO_ID,
        repo_type="dataset",
    )
    api.upload_file(
        path_or_fileobj=str(dataset_card_path),
        path_in_repo="README.md",
        repo_id=CORPUS_REPO_ID,
        repo_type="dataset",
    )
    print(f"アップロード完了: https://huggingface.co/datasets/{CORPUS_REPO_ID}")
else:
    print(
        "アップロードをスキップした(DRY_RUN=True またはローカル実行のため)。"
        "本番実行は Google Colab で DRY_RUN=False の状態で行うこと。"
    )

アップロードをスキップした(DRY_RUN=True またはローカル実行のため)。本番実行は Google Colab で DRY_RUN=False の状態で行うこと。


## まとめ

- コーパスを本番スケール(1000 記事のマニフェスト全体)のままローカルで取得した。
- `_fetch_wikipedia_revision_plaintext()`・`load_wikipedia_corpus()`の修正により、
  記事単位の取得失敗が全体を止めないことを確認した(取得できた記事数・スキップした
  記事とその理由はセル出力を参照)。
- `split_train_val_text()`が決定的であることを確認した(009 本体側の分割結果と
  完全に一致する)。
- アップロード用の`corpus.json`・データセットカードを`.cache/`に書き出した。
- **アップロードは実行していない。** こうじさんが Google Colab で `DRY_RUN=False`
  に切り替えて実行する必要がある。